# CapitalTrust EGP Financial Analyst - System Execution
**Evaluator-Optimizer multi-agent system (LangGraph) with a mandatory Human-in-the-Loop pre-publish gate.**

Sections: 1 Setup · 2 CBE ingestion + audited RAG · 3 EGX context · 4 MCP tool + `bind_tools` · 5 A2A schema compliance · 6 Full run: reject → revise → approve → **pause** · 7 Memory trimming · 8 Process restart + resume · 9 Audit trail · 10 Reject path & guard-rails · 11 Acceptance checklist

> **Run mode.** If `GROQ_API_KEY` is set (in `deliverables/.env` or the environment) the Analyst/Reviewer use a real Groq model (`GROQ_MODEL`, default `openai/gpt-oss-120b`); otherwise a deterministic *offline stand-in* is used (it is not a language model - see `OfflineDemoChatModel`). Embeddings are local (`fastembed`), because Groq has no embeddings API. Real Langfuse trace links are produced when `LANGFUSE_PUBLIC_KEY`/`LANGFUSE_SECRET_KEY` are set; otherwise a `local-audit://` fallback link into the hash-chained audit log is printed. The notebook states the active mode below.


## 1. Setup

In [1]:
import sys, os, json, shutil, subprocess, contextlib, io, difflib
from pathlib import Path
SRC = Path("../src").resolve()
sys.path.insert(0, str(SRC))
try:                                   # pick up GROQ_API_KEY / LANGFUSE_* from deliverables/.env
    from dotenv import load_dotenv; load_dotenv("../.env")
except ImportError:
    pass

import workflow_graph as wg
from state_schema import A2AMessage, ReviewVerdict, HumanDecision
from mcp_tools import list_mcp_tools, fetch_egp_rate_via_mcp, get_exchange_rate_tool
import pandas as pd
from IPython.display import display, Markdown

shutil.rmtree(wg.RUNTIME_DIR, ignore_errors=True)       # clean, reproducible run (vector store, sqlite, audit log)
llm, MODE = wg.get_llm()
audit = wg.AuditLog()
auditor = wg.RetrievalAuditor(audit)
print("LLM mode        :", MODE, "->", type(llm).__name__, getattr(llm, "model_name", ""))
if MODE == "offline":
    print("                  (no GROQ_API_KEY found -> deterministic offline stand-in; add the key to ../.env for a real LLM)")
print("Embeddings      :", wg.get_embeddings()[1])
print("Tracer          :", auditor.mode)
print("Checkpointer    : SqliteSaver at", wg.DEFAULT_DB.relative_to(wg.ROOT))
print("Audit log       :", wg.DEFAULT_AUDIT.relative_to(wg.ROOT))

LLM mode        : online -> ChatGroq openai/gpt-oss-120b
Embeddings      : fastembed
Tracer          : langfuse
Checkpointer    : SqliteSaver at runtime\checkpoints.sqlite
Audit log       : runtime\audit_log.jsonl


## 2. Ingest the CBE guidelines into Chroma, then query with an audited retriever
`TextLoader` → `MarkdownHeaderTextSplitter` (keeps each numbered clause intact) → `RecursiveCharacterTextSplitter` → Chroma (persistent). Chunk metadata carries circular ID, issue date, section and the document SHA-256 for auditability.

In [2]:
vs = wg.build_vectorstore(rebuild=True)
chunks = wg.load_and_chunk_cbe()
print(f"{len(chunks)} chunks ingested into Chroma collection '{vs._collection.name}' ({vs._collection.count()} vectors)\n")
display(pd.DataFrame([{"idx": c.metadata["chunk_index"], "section": c.metadata["section_label"][:55],
                       "chars": len(c.page_content), "circular": c.metadata["circular_id"]} for c in chunks]).head(10))

24 chunks ingested into Chroma collection 'cbe_guidelines_fastembed' (24 vectors)



,idx,section,chars,circular
0,0,MONETARY POLICY AND REGULATORY FRAMEWORK GUIDE...,497,CBE-MPD-2023-049
1,1,PREAMBLE AND MACROECONOMIC CONTEXT,866,CBE-MPD-2023-049
2,2,PREAMBLE AND MACROECONOMIC CONTEXT,228,CBE-MPD-2023-049
3,3,1.1 Current Corridor Rates,642,CBE-MPD-2023-049
4,4,1.2 Rationale and Transmission Mechanism,768,CBE-MPD-2023-049
5,5,1.3 Forward Guidance,313,CBE-MPD-2023-049
6,6,2.1 Baseline Target and Tolerance Bands,399,CBE-MPD-2023-049
7,7,2.2 Core vs. Headline Inflation Dynamics,673,CBE-MPD-2023-049
8,8,3.1 Exchange Rate Regime,497,CBE-MPD-2023-049
9,9,3.2 FX Liquidity Management and Interbank Prot...,631,CBE-MPD-2023-049


In [3]:
retriever = vs.as_retriever(search_kwargs={"k": 2})
docs, info = auditor.retrieve(retriever, "What are the CBE overnight deposit and lending rates?", "demo-retrieval", 1)
for d in docs:
    print("\n---", d.metadata["section_label"], "| circular", d.metadata["circular_id"])
    print(d.page_content[:350])
print("\nTRACE LINK:", info["trace_link"])

[RAG #1] 'What are the CBE overnight deposit and lending rates?' -> 2 chunks | trace: https://cloud.langfuse.com/project/cmwmxhw4f013cad0g6gfv2seq/traces/98c1d1aafbfdb21b8e0a6524370724a9

--- 1.1 Current Corridor Rates | circular CBE-MPD-2023-049
### SECTION 1: INTEREST RATE DECISIONS AND MONETARY POLICY STANCE  
#### 1.1 Current Corridor Rates
Following the latest assembly of the Monetary Policy Committee (MPC), the CBE has evaluated current domestic liquidity constraints and core inflation trajectories. To anchor inflation expectations and mitigate second-round effects of recent supply sh

--- 5.2 Deposit Auctions | circular CBE-MPD-2023-049
#### 5.2 Deposit Auctions
The CBE conducts fixed-rate and variable-rate deposit auctions on a weekly basis to absorb structural excess liquidity. Financial institutions must closely monitor the accepted cutoff yields at these auctions, as they serve as the most accurate leading indicator of short-term Egyptian pound interbank rates (CAIBOR).  
--

## 3. EGX market context from `egx_daily_summaries.csv`

In [4]:
egx = wg.load_egx_context()
print(f"{egx['sessions']} sessions: {egx['first_date']} -> {egx['latest_date']}   (only {egx['sessions']} sessions exist in the file)")
display(pd.DataFrame(egx["moves"]).T[["sector","prev_close","close","pct_change","volume","avg_prior_volume","vol_vs_prior_pct"]])
print("Equal-weighted basket day-on-day change:", egx["basket_avg_pct"], "%")

3 sessions: 2024-03-05 -> 2024-03-07   (only 3 sessions exist in the file)


,sector,prev_close,close,pct_change,volume,avg_prior_volume,vol_vs_prior_pct
ABUK,"Basic Resources, Chemicals",88.5,85.0,-4.0,900000,1030000,-12.6
COMI,"Financial Services, Banking",83.0,80.5,-3.0,4100000,3552250,15.4
EAST,"Consumer Goods, Tobacco",31.5,30.2,-4.1,1800000,1572500,14.5
FWRY,"Information Technology, Fintech",5.6,5.45,-2.7,4500000,5850350,-23.1
HRHO,"Financial Services, Investment",18.9,18.2,-3.7,2200000,2251000,-2.3
ORAS,"Construction, Materials",165.0,160.0,-3.0,320000,285000,12.3
SWDY,"Industrial Goods, Utilities",35.2,34.1,-3.1,1600000,1525000,4.9
TMGH,"Real Estate, Development",29.8,28.5,-4.4,4100000,5450050,-24.8


Equal-weighted basket day-on-day change: -3.5 %


## 4. MCP exchange-rate tool, bound to the Analyst with `bind_tools`
`mcp_egp_server.py` is launched as a real MCP stdio server; `mcp_tools.py` performs `tools/list` and `tools/call` through the MCP client.

In [5]:
print("tools/list over MCP:"); print(json.dumps(list_mcp_tools(), indent=1)[:600])
print("\ntools/call fetch_egp_rate(USD):", fetch_egp_rate_via_mcp("USD"))
try:
    fetch_egp_rate_via_mcp("XYZ")
except Exception as e:
    print("unsupported currency is rejected ->", type(e).__name__, str(e)[:90])
bound = llm.bind_tools([get_exchange_rate_tool()])
resp = bound.invoke("Get the current USD rate.")
print("\nbind_tools -> model emitted tool call:", resp.tool_calls)

tools/list over MCP:
[
 {
  "name": "fetch_egp_rate",
  "description": "Retrieves the current exchange rate for the Egyptian Pound (EGP) against the specified base currency.",
  "input_schema": {
   "properties": {
    "base_currency": {
     "default": "USD",
     "title": "Base Currency",
     "type": "string"
    }
   },
   "title": "fetch_egp_rateArguments",
   "type": "object"
  }
 }
]

tools/call fetch_egp_rate(USD): {'base': 'USD', 'target': 'EGP', 'rate': 47.85, 'source': 'CBE_MOCK', 'last_updated': '2026-11-19T02:59:33+00:00'}
unsupported currency is rejected -> RuntimeError MCP tool fetch_egp_rate failed: Error executing tool fetch_egp_rate: No mock quote for XYZ

bind_tools -> model emitted tool call: [{'name': 'fetch_egp_rate', 'args': {'base_currency': 'USD'}, 'id': 'fc_d7c8e075-aa78-466a-ba4c-2c1a212c0778', 'type': 'tool_call'}]


## 5. Schema compliance (A2A)
Every hand-off is validated by the Pydantic `A2AMessage`, and its payload is also validated against the published `schemas/a2a_message_schema.json` (draft-07).

In [6]:
import jsonschema
schema = json.loads((wg.ROOT / "schemas" / "a2a_message_schema.json").read_text())
jsonschema.Draft7Validator.check_schema(schema)

# (a) the example shipped inside the JSON schema must parse with our Pydantic model
ex = schema["examples"][0]
m = A2AMessage(sender="reviewer", recipient="analyst", message_type="review_feedback", iteration=1, **ex)
jsonschema.validate(m.schema_payload(), schema)
print("schema example -> Pydantic OK and JSON-Schema OK")
schema_ok = True; negatives_rejected = 0

# (b) negative tests: malformed hand-offs must be rejected
from pydantic import ValidationError
bad = {"sender":"analyst","recipient":"reviewer","message_type":"draft_submission","iteration":1,"draft_report":"x"}
for label, patch in [("empty sources_cited", {"sources_cited": []}), ("blank source", {"sources_cited": [" "]}),
                     ("unknown field", {"sources_cited": ["a"], "bogus": 1}), ("iteration 0", {"sources_cited": ["a"], "iteration": 0})]:
    try:
        A2AMessage(**{**bad, **patch}); print("!! accepted", label)
    except ValidationError as e:
        negatives_rejected += 1
        print(f"rejected ({label}):", e.errors()[0]["msg"][:70])

schema example -> Pydantic OK and JSON-Schema OK
rejected (empty sources_cited): List should have at least 1 item after validation, not 0
rejected (blank source): Value error, sources_cited must not contain blank entries
rejected (unknown field): Extra inputs are not permitted
rejected (iteration 0): Input should be greater than or equal to 1


In [7]:
# Reviewer rules in isolation: a tampered draft must be caught by the hard checks
facts = {"circular_id":"CBE-MPD-2023-049","circular_issued":"November 12, 2023",
         "policy_rates":{"overnight_deposit":19.25,"overnight_lending":20.25,"main_operation":19.75,"discount":19.75},
         "fx":{"base":"USD","rate":47.85,"source":"CBE_MOCK"}, "egx": egx}
tampered = ("DRAFT. Pending human director approval.\nPer CBE Circular CBE-MPD-2023-049 (issued November 12, 2023) the overnight deposit rate is 18.25%.\n"
            "The EGP/USD exchange rate is 49.10 (mock).\nCOMI rose 12% on 2024-03-07. We guarantee gains - strong buy recommendation.")
for i in wg.run_regulatory_checks(tampered, ["CBE Circular CBE-MPD-2023-049", "Bloomberg"], facts):
    print(f"[{i['rule']}] {i['problem'][:110]}")

[R2] Sentence mentions the overnight deposit but not the circular's figure 19.25%: 'Per CBE Circular CBE-MPD-2023-0
[R3] The tool rate 47.85 EGP/USD is not stated.
[R3] Quotes 49.10 for the exchange rate; the MCP tool returned 47.85.
[R4] Percentage 12% in 'COMI rose 12% on 2024-03-07....' does not match any figure derivable from the EGX CSV.
[R6] Prohibited promotional / directive language: 'We guarantee gains - strong buy recommendation.'
[R8] Unrecognised / unverifiable source: 'Bloomberg'


## 6. Full run: Analyst → Reviewer ✗ → Analyst (revises) → Reviewer ✓ → **Pre-publish interrupt**
`stress_first_draft=True` is a demo/test switch (off by default): after the LLM writes its *first* draft, three known faults are injected deterministically (wrong circular number, unsupported 30-day claim, unlabelled speculation) so the Reviewer's rejection is shown on demand even with a well-behaved model; the Analyst must then repair them from the feedback. It is a no-op in offline mode (the stand-in always does this). `history_window=4` is used here so the sliding-window/summary step is visible in a short run. Each RAG retrieval prints its trace link as it happens.

In [8]:
DB = wg.DEFAULT_DB
THREAD = "capitaltrust-brief-2024-03-08-A"
graph = wg.build_graph(wg.make_checkpointer(DB), llm=llm, mode=MODE, vectorstore=vs, audit_log=audit, history_window=4,
                     stress_first_draft=True)   # demo flag: inject known faults into draft 1 so the Reviewer has something to reject
cfg = wg.thread_config(THREAD)

drafts, interrupt_payload = [], None
for ev in graph.stream(wg.initial_input(THREAD), cfg, stream_mode="updates"):
    for node, upd in ev.items():
        if node == "__interrupt__":
            interrupt_payload = upd[0].value
            print("\n⏸  GRAPH SUSPENDED at human_gate (interrupt) - waiting for a human director\n"); continue
        if node == "analyst":
            drafts.append(upd["draft_report"])
            print(f"\n▶ ANALYST draft #{upd['iteration']}  ({len(upd['draft_report'])} chars; A2A {upd['a2a_message']['sender']}->{upd['a2a_message']['recipient']})")
        elif node == "reviewer":
            r = upd["review"]
            print(f"▶ REVIEWER is_approved={r['is_approved']}  judge={r['judge']}")
            print("   " + r["feedback_notes"].replace("\n", "\n   ")[:1400])
        elif node == "memory_manager":
            print("▶ MEMORY  trimmed history ->", upd.get("history_summary", "(nothing to trim)"))

[RAG #1] 'CBE policy rates overnight deposit rate overnight lending ' -> 2 chunks | trace: https://cloud.langfuse.com/project/cmwmxhw4f013cad0g6gfv2seq/traces/0a99421c07a7e310f8bd2434df37bd64
[RAG #1] 'exchange rate regime EGP USD foreign exchange directives' -> 2 chunks | trace: https://cloud.langfuse.com/project/cmwmxhw4f013cad0g6gfv2seq/traces/e1c7272ae127b3a26150a63ff7798ad0
[RAG #1] 'AI automated advisory human-in-the-loop pre-publish audit ' -> 2 chunks | trace: https://cloud.langfuse.com/project/cmwmxhw4f013cad0g6gfv2seq/traces/ab4350c58399602e46743c9e0dd74671

▶ ANALYST draft #1  (1699 chars; A2A analyst->reviewer)
▶ REVIEWER is_approved=False  judge=openai/gpt-oss-120b
   REJECTED. Revise the draft; fix every item and do not introduce new unsupported claims:
   [R1] The draft does not cite the circular ID CBE-MPD-2023-049.
   [R1] Cites 'Circular No. 45/2023', which is not in the retrieved CBE material; the retrieved circular is CBE-MPD-2023-049.
   [R4] References a 30-day / 

In [9]:
hist = graph.get_state(cfg).values["review_history"]
display(pd.DataFrame(hist)[["iteration","is_approved","rules_failed","judge"]])
# ---- Acceptance: self-correction ----
assert hist[0]["is_approved"] is False and hist[-1]["is_approved"] is True and len(hist) >= 2
print("✅ SELF-CORRECTION VERIFIED: rejected at iteration 1 (rules %s) -> revised -> approved at iteration %d" % (hist[0]["rules_failed"], hist[-1]["iteration"]))
print("\nWhat changed between draft 1 and draft 2:")
for l in difflib.unified_diff(drafts[0].splitlines(), drafts[-1].splitlines(), "draft_1", "draft_2", lineterm="", n=0):
    if not l.startswith(("@@", "---", "+++")) and "as of" not in l: print("  ", l[:150])

,iteration,is_approved,rules_failed,judge
0,1,False,"[R1, R4, R5]",openai/gpt-oss-120b
1,2,True,[],openai/gpt-oss-120b


✅ SELF-CORRECTION VERIFIED: rejected at iteration 1 (rules ['R1', 'R4', 'R5']) -> revised -> approved at iteration 2

What changed between draft 1 and draft 2:
   -According to CBE circular CBE Circular No. 45/2023 issued November 12, 2023, the Overnight Deposit Rate is set at 19.25%.  
   -The Overnight Lending Rate is 20.25% as per the same circular.  
   -The Rate of the Main Operation stands at 19.75%.  
   -The Discount Rate is also 19.75%.  
   +According to CBE-MPD-2023-049 issued November 12, 2023, the Overnight Deposit Rate is 19.25%.  
   +According to CBE-MPD-2023-049 issued November 12, 2023, the Overnight Lending Rate is 20.25%.  
   +According to CBE-MPD-2023-049 issued November 12, 2023, the Rate of the Main Operation is 19.75%.  
   +According to CBE-MPD-2023-049 issued November 12, 2023, the Discount Rate is 19.75%.  
   -The Central Bank maintains a flexible exchange rate regime, and the current EGP/USD rate is 47.85 per USD, source: CBE_MOCK (mock).  
   +The Central

### Pre-publish hold state (nothing has been released)

In [10]:
snap = graph.get_state(cfg)
print("next node        :", snap.next)
print("status           :", snap.values["status"])
pre_resume_published = snap.values.get("published_report"); pre_status = snap.values["status"]
live_msgs_valid = all(jsonschema.validate(A2AMessage.model_validate(snap.values["a2a_message"]).schema_payload(), schema) is None for _ in [0])
print("published_report :", repr(pre_resume_published), "(empty - nothing published)")
print("reviewer approved:", interrupt_payload["reviewer_approved"], "| iterations used:", interrupt_payload["iterations_used"])
print("\nHuman instructions:", interrupt_payload["instructions"])
display(Markdown("**Draft awaiting the director's decision:**\n\n```\n" + interrupt_payload["draft_report"] + "\n```"))

next node        : ('human_gate',)
status           : pre_publish
published_report : None (empty - nothing published)
reviewer approved: True | iterations used: 2

Human instructions: Resume with {'action': 'publish'|'reject', 'reviewer': '<name>', 'comment': '...'}.


**Draft awaiting the director's decision:**

```
DRAFT - Pending human director approval  

According to CBE-MPD-2023-049 issued November 12, 2023, the Overnight Deposit Rate is 19.25%.  
According to CBE-MPD-2023-049 issued November 12, 2023, the Overnight Lending Rate is 20.25%.  
According to CBE-MPD-2023-049 issued November 12, 2023, the Rate of the Main Operation is 19.75%.  
According to CBE-MPD-2023-049 issued November 12, 2023, the Discount Rate is 19.75%.  

The Central Bank reiterates its commitment to a durably flexible exchange rate regime.  
The current EGP/USD rate is 47.85, source: CBE_MOCK (mock).  

On the latest EGX session dated 2024-03-07:  
- ABUK closed at 85.00, down 4.0% day-on-day, volume 900,000 (-12.6% vs mean of prior sessions).  
- COMI closed at 80.50, down 3.0% day-on-day, volume 4,100,000 (+15.4% vs mean of prior sessions).  
- EAST closed at 30.20, down 4.1% day-on-day, volume 1,800,000 (+14.5% vs mean of prior sessions).  
- FWRY closed at 5.45, down 2.7% day-on-day, volume 4,500,000 (-23.1% vs mean of prior sessions).  
- HRHO closed at 18.20, down 3.7% day-on-day, volume 2,200,000 (-2.3% vs mean of prior sessions).  
- ORAS closed at 160.00, down 3.0% day-on-day, volume 320,000 (+12.3% vs mean of prior sessions).  
- SWDY closed at 34.10, down 3.1% day-on-day, volume 1,600,000 (+4.9% vs mean of prior sessions).  
- TMGH closed at 28.50, down 4.4% day-on-day, volume 4,100,000 (-24.8% vs mean of prior sessions).  

The equal-weighted basket recorded a day-on-day decline of -3.5%.  

Analyst Speculation: Real estate names such as TMGH may experience short-term volatility until secondary CBE guidelines are released.  

sources_cited: [CBE-MPD-2023-049, EGX daily summary (2024-03-05 to 2024-03-07), MCP tool fetch_egp_rate]
```

## 7. Context-window control: sliding window + summarisation
`compact_history` keeps the last *N* messages verbatim and replaces everything older with a running summary (it never separates a `ToolMessage` from its tool-calling `AIMessage`). Below: the live thread, then a synthetic 40-message history.

In [11]:
v = graph.get_state(cfg).values
print("live thread: %d messages kept (window=4 + latest cycle) | summary: %s" % (len(v["messages"]), v["history_summary"]))
from langchain_core.messages import HumanMessage, AIMessage
long = [HumanMessage(content="task", name="cio", id="m0")]
for i in range(1, 40):
    long.append((HumanMessage(content=f"[A2A reviewer->analyst] [R{i%5+1}] issue", id=f"m{i}") if i % 2 else AIMessage(content=f"draft {i}", id=f"m{i}")))
upd = wg.compact_history(long, "", llm, MODE, window=8)
print("synthetic: %d messages -> remove %d, keep %d" % (len(long), len(upd["messages"]), len(long)-len(upd["messages"])))
print("summary   :", upd["history_summary"])

live thread: 8 messages kept (window=4 + latest cycle) | summary: 2 earlier messages compacted (0 drafts; original task: Draft the CapitalTrust morning brief for market open: CBE policy stance, EGP/USD rate and ).
synthetic: 40 messages -> remove 32, keep 8
summary   : 32 earlier messages compacted (15 drafts; original task: task; reviewer rules raised: R1, R2, R3, R4, R5).


## 8. Process restart while paused → resume with the persistent thread ID
The checkpointer is a **SQLite** file (`MemorySaver` is in-memory only and would lose the paused thread). We drop every in-memory object, then start a **brand-new Python process** that rebuilds the graph from disk and resumes the same `thread_id` with the human decision.

In [12]:
import sqlite3
Path(wg.RUNTIME_DIR / "paused_thread_id.txt").write_text(THREAD)
del graph, vs, retriever, auditor, audit, snap, v           # forget everything in this kernel
con = sqlite3.connect(DB); n = con.execute("select count(*) from checkpoints where thread_id=?", (THREAD,)).fetchone()[0]; con.close()
print(f"checkpoint rows for thread {THREAD!r} in {DB.name}: {n}")

# --- peek (new process #1): thread is still suspended at the gate
peek = subprocess.run([sys.executable, "-c", f"""
import sys; sys.path.insert(0, {str(SRC)!r})
import workflow_graph as wg
g = wg.build_graph(wg.make_checkpointer(wg.DEFAULT_DB))
s = g.get_state(wg.thread_config({THREAD!r}))
print('NEW PROCESS pid-fresh | next =', s.next, '| status =', s.values['status'], '| published =', repr(s.values.get('published_report')))
"""], capture_output=True, text=True, encoding="utf-8", errors="replace", env={**os.environ, "PYTHONIOENCODING": "utf-8"}); print(peek.stdout.strip(), peek.stderr.strip()[-300:])

checkpoint rows for thread 'capitaltrust-brief-2024-03-08-A' in checkpoints.sqlite: 7
NEW PROCESS pid-fresh | next = ('human_gate',) | status = pre_publish | published = None 


In [13]:
# --- the human director's decision, delivered in yet another fresh process
res = subprocess.run([sys.executable, "-c", f"""
import sys, json; sys.path.insert(0, {str(SRC)!r})
import workflow_graph as wg
vals = wg.resume_thread({THREAD!r}, "publish", reviewer="Youssef El-Menshawy (CIO)", comment="Reviewed - release at market open")
print(json.dumps({{'status': vals['status'], 'decision': vals['human_decision']}}, indent=1))
print('-----'); print(vals['published_report'][-260:])
"""], capture_output=True, text=True, encoding="utf-8", errors="replace", env={**os.environ, "PYTHONIOENCODING": "utf-8"})
print(res.stdout); print(res.stderr[-400:] if res.returncode else "")
assert '"status": "published"' in res.stdout
print("✅ PERSISTENCE VERIFIED: paused thread survived a process restart and completed on resume")

{
 "status": "published",
 "decision": {
  "action": "publish",
  "reviewer": "Youssef El-Menshawy (CIO)",
  "comment": "Reviewed - release at market open"
 }
}
-----
until secondary CBE guidelines are released.  

sources_cited: [CBE-MPD-2023-049, EGX daily summary (2024-03-05 to 2024-03-07), MCP tool fetch_egp_rate]

Released by Youssef El-Menshawy (CIO) at 2026-11-19T03:02:07+00:00 (human approval recorded in audit log).


✅ PERSISTENCE VERIFIED: paused thread survived a process restart and completed on resume


In [14]:
# resuming a finished thread must fail (no double publish)
again = subprocess.run([sys.executable, "-c", f"""
import sys; sys.path.insert(0, {str(SRC)!r})
import workflow_graph as wg
wg.resume_thread({THREAD!r}, "publish")"""], capture_output=True, text=True, encoding="utf-8", errors="replace", env={**os.environ, "PYTHONIOENCODING": "utf-8"})
print("second resume ->", again.stderr.strip().splitlines()[-1][:160])

second resume -> RuntimeError: Thread 'capitaltrust-brief-2024-03-08-A' is not waiting for input (next=(), status=published).


## 9. Audit trail: every RAG retrieval, with its trace link

In [15]:
audit = wg.AuditLog()
graph = wg.build_graph(wg.make_checkpointer(DB), llm=llm, mode=MODE, vectorstore=wg.build_vectorstore(), audit_log=audit)
final = graph.get_state(cfg).values
links = pd.DataFrame(final["audit_links"])
links["query"] = links["query"].str[:48]
print(f"{len(links)} RAG retrievals recorded for thread {THREAD} (tracer: {links.tracer.iloc[0]}):")
for _, r in links.iterrows():
    print(f"  iter {r.iteration} | {r['query']:<48} | {r.trace_link}")
rows = audit.entries(THREAD)
ok, n = audit.verify()
print(f"\nhash-chain verification over entire audit log: ok={ok}, entries={n}")
display(pd.DataFrame([{"ts": r["ts"], "event": r["event"], "detail": (r["payload"].get("action") or r["payload"].get("query","")[:40] or r["payload"].get("is_approved",""))} for r in rows]).tail(9))
assert len(links) == 3 * final["iteration"] and ok

6 RAG retrievals recorded for thread capitaltrust-brief-2024-03-08-A (tracer: langfuse):
  iter 1 | CBE policy rates overnight deposit rate overnigh | https://cloud.langfuse.com/project/cmwmxhw4f013cad0g6gfv2seq/traces/0a99421c07a7e310f8bd2434df37bd64
  iter 1 | exchange rate regime EGP USD foreign exchange di | https://cloud.langfuse.com/project/cmwmxhw4f013cad0g6gfv2seq/traces/e1c7272ae127b3a26150a63ff7798ad0
  iter 1 | AI automated advisory human-in-the-loop pre-publ | https://cloud.langfuse.com/project/cmwmxhw4f013cad0g6gfv2seq/traces/ab4350c58399602e46743c9e0dd74671
  iter 2 | CBE policy rates overnight deposit rate overnigh | https://cloud.langfuse.com/project/cmwmxhw4f013cad0g6gfv2seq/traces/743cf17cd96d7157a97574e318dfa5f0
  iter 2 | exchange rate regime EGP USD foreign exchange di | https://cloud.langfuse.com/project/cmwmxhw4f013cad0g6gfv2seq/traces/2183a9f5fb4f04d624c4ab0cdb28bbb1
  iter 2 | AI automated advisory human-in-the-loop pre-publ | https://cloud.langfuse.com/project

,ts,event,detail
2,2026-11-19T02:59:44+00:00,rag_retrieval,AI automated advisory human-in-the-loop
3,2026-11-19T02:59:55+00:00,analyst_draft,
4,2026-11-19T02:59:57+00:00,reviewer_verdict,False
5,2026-11-19T02:59:58+00:00,rag_retrieval,CBE policy rates overnight deposit rate
6,2026-11-19T02:59:58+00:00,rag_retrieval,exchange rate regime EGP USD foreign exc
7,2026-11-19T02:59:59+00:00,rag_retrieval,AI automated advisory human-in-the-loop
8,2026-11-19T03:00:43+00:00,analyst_draft,
9,2026-11-19T03:01:00+00:00,reviewer_verdict,True
10,2026-11-19T03:02:07+00:00,human_decision,publish


## 10. Reject path and guard-rails
* **Reject:** the director can refuse; nothing is published.
* **Escalation guard:** if the Reviewer never approves within `max_iterations`, the thread still pauses for a human - but publishing is *blocked*; only a rejection is honoured.

In [16]:
def run_to_gate(thread, **kw):
    with contextlib.redirect_stdout(io.StringIO()):
        g = wg.build_graph(wg.make_checkpointer(DB), llm=llm, mode=MODE, vectorstore=wg.build_vectorstore(), audit_log=audit, **kw)
        list(g.stream(wg.initial_input(thread), wg.thread_config(thread), stream_mode="updates"))
    return g
def decide(g, thread, action):
    with contextlib.redirect_stdout(io.StringIO()):
        from langgraph.types import Command
        g.invoke(Command(resume={"action": action, "reviewer": "Compliance Officer"}), wg.thread_config(thread))
    return g.get_state(wg.thread_config(thread)).values

g2 = run_to_gate("brief-B-reject")
v2 = decide(g2, "brief-B-reject", "reject")
print("Thread B  director rejects      ->", v2["status"], "| published_report:", repr(v2["published_report"]))

g3 = run_to_gate("brief-C-escalated", max_iterations=1, stress_first_draft=True)
print("Thread C  reviewer approved?    ->", g3.get_state(wg.thread_config("brief-C-escalated")).values["review"]["is_approved"], "(max_iterations=1 reached, escalated)")
v3 = decide(g3, "brief-C-escalated", "publish")
print("Thread C  human tries 'publish' ->", v3["status"], "| published_report:", repr(v3["published_report"]))
assert v2["status"] == "rejected" and v3["status"] == "blocked"

Thread B  director rejects      -> rejected | published_report: ''
Thread C  reviewer approved?    -> False (max_iterations=1 reached, escalated)
Thread C  human tries 'publish' -> blocked | published_report: ''


### Trace links for the reject / escalation threads
The cell above silences stdout while running threads B and C, so their retrieval links are printed here (they are stored in each thread's `audit_links`).

In [17]:
extra_links = []
for g_, t_ in [(g2, "brief-B-reject"), (g3, "brief-C-escalated")]:
    for l in g_.get_state(wg.thread_config(t_)).values["audit_links"]:
        extra_links.append({**l, "thread": t_})
        print(f"{t_} | iter {l['iteration']} | {l['query'][:48]:<48} | {l['trace_link']}")
print(f"\n{len(extra_links)} additional retrievals, tracer: {extra_links[0]['tracer'] if extra_links else '-'}")

brief-B-reject | iter 1 | CBE policy rates overnight deposit rate overnigh | https://cloud.langfuse.com/project/cmwmxhw4f013cad0g6gfv2seq/traces/c7568ec800d687be8c0d00d6f2c63b7a
brief-B-reject | iter 1 | exchange rate regime EGP USD foreign exchange di | https://cloud.langfuse.com/project/cmwmxhw4f013cad0g6gfv2seq/traces/33efb00cc84e28d547a49d5f534de1e7
brief-B-reject | iter 1 | AI automated advisory human-in-the-loop pre-publ | https://cloud.langfuse.com/project/cmwmxhw4f013cad0g6gfv2seq/traces/9ada389c025565ab0b72222efaf27e7c
brief-C-escalated | iter 1 | CBE policy rates overnight deposit rate overnigh | https://cloud.langfuse.com/project/cmwmxhw4f013cad0g6gfv2seq/traces/f92c1eb06915eb09e85b9ae0863d1c32
brief-C-escalated | iter 1 | exchange rate regime EGP USD foreign exchange di | https://cloud.langfuse.com/project/cmwmxhw4f013cad0g6gfv2seq/traces/efe1793b2bdb8ad9031e739fe82416d7
brief-C-escalated | iter 1 | AI automated advisory human-in-the-loop pre-publ | https://cloud.langfuse.c

## 11. Acceptance checklist

In [18]:
ok, n = audit.verify()
checks = {
 "Self-correction: reject -> revise -> approve (thread A)": hist[0]["is_approved"] is False and hist[-1]["is_approved"] is True,
 "Pre-publish interrupt reached; nothing published before human action": interrupt_payload is not None and not pre_resume_published and pre_status == "pre_publish",
 "State persistence: paused thread resumed from SQLite in a fresh process": '"status": "published"' in res.stdout,
 "Schema compliance: example + live messages validate; 4/4 malformed rejected": schema_ok and negatives_rejected == 4 and live_msgs_valid,
 f"Auditability: {len(links)} retrievals each with a trace link ({links.tracer.iloc[0]})": links.trace_link.notna().all(),
 "Audit log hash-chain intact": ok,
 "Auditability (threads B and C): every retrieval also has a trace link": len(extra_links) > 0 and all(l["trace_link"] for l in extra_links),
 "Reject / blocked-publish paths behave": v2["status"] == "rejected" and v3["status"] == "blocked",
}
for k, val in checks.items(): print("✅" if val else "❌", k)
assert all(checks.values())

✅ Self-correction: reject -> revise -> approve (thread A)
✅ Pre-publish interrupt reached; nothing published before human action
✅ State persistence: paused thread resumed from SQLite in a fresh process
✅ Schema compliance: example + live messages validate; 4/4 malformed rejected
✅ Auditability: 6 retrievals each with a trace link (langfuse)
✅ Audit log hash-chain intact
✅ Auditability (threads B and C): every retrieval also has a trace link
✅ Reject / blocked-publish paths behave
